# Brain Surgery — Bước 2 (E4B): QLoRA SFT -> ORPO & Export GGUF 4GB VRAM

Notebook chạy trọn gói trên **Google Colab Pro (L4 24GB hoặc A100 40GB)** cho checkpoint **`haku-nekooo/gemma4-e4b-nuclear-pruned`** và đẩy thành phẩm lên **`haku-nekooo/gemma4-e4b-nuclear-agent`**:
1. **Dữ liệu linh hoạt**: Hỗ trợ cả upload trực tiếp lên `/content/data_distill` lẫn mount **Google Drive** (`sft_gemma4.jsonl` + `orpo_gemma4.jsonl` hoặc `sft_final_1350.jsonl` + `pref_final_270.jsonl`), tự động kiểm đo độ dài token thật + `0 <unk>`.
2. **Stage 2A (SFT)**: Đóng băng Vision/Audio (`requires_grad=False`), train LoRA Attention/MLP (`lr=1.5e-4`) + `embed_tokens` / `embed_tokens_per_layer` / `lm_head` (`embedding_lr=1.5e-5`), assistant-only loss masking.
3. **Stage 2B (ORPO 1 epoch)**: Khóa cứng cả Vision/Audio và embeddings/PLE, tinh chỉnh `beta=0.1, lr=8e-6` trị các lỗi `nc-agent` + PER Loop.
4. **Export GGUF tối ưu 4GB VRAM (GTX 1650 Ti)**: `IQ4_XS` + `Q4_K_M` (kèm `--imatrix` từ chính tập dữ liệu) + `mmproj` bản `bf16` & `q8_0` (chạy đồng thời với `-ot "per_layer_token_embd\.weight=CPU"`).

## 1. Cấu hình Pipeline, Đường dẫn Dữ liệu & Đồng bộ Repo

In [ ]:
from pathlib import Path

BASE_HF_REPO = "haku-nekooo/gemma4-e4b-nuclear-pruned"
TARGET_HF_REPO = "haku-nekooo/gemma4-e4b-nuclear-agent"
GITHUB_REPO = "https://github.com/just-haku/brain-surgery.git"

WORK_DIR = Path("/content/work_step2")
REPO_DIR = Path("/content/brain-surgery")
LLAMA_CPP_DIR = Path("/content/llama.cpp")

# Chọn nguồn dữ liệu: "colab_upload" | "google_drive" | "hf_dataset"
DATA_SOURCE_MODE = "colab_upload"
UPLOAD_DATA_DIR = Path("/content/data_distill")
DRIVE_DATA_DIR = Path("/content/drive/MyDrive/brain_surgery/distill")
DATA_HF_REPO = ""  # Nếu dùng "hf_dataset", điền ví dụ "haku-nekooo/nuclear-code-distill"

# Sequence length đã kiểm định trên 1,620 mẫu (max ~3,122 tok SFT / ~2,270 tok ORPO)
SFT_MAX_SEQ_LENGTH = 8192
ORPO_MAX_PROMPT_LENGTH = 2560
ORPO_MAX_LENGTH = 4096

RUN_SFT = True
RUN_ORPO = True
TRAIN_EMBEDDINGS = True

USE_IMATRIX = True
IMATRIX_SAMPLES = 128
EXPORT_QUANTS = ["IQ4_XS", "Q4_K_M"]
MMPROJ_OUTTYPES = ["bf16", "q8_0"]
PUSH_TO_HF = True

In [ ]:
import os
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--upgrade",
        "transformers>=5.5.0",
        "peft>=0.15.0",
        "trl>=0.16.0",
        "datasets>=3.0.0",
        "bitsandbytes>=0.45.0",
        "accelerate>=1.3.0",
        "huggingface_hub>=0.28.0",
        "gguf",
    ],
    check=True,
)

if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", GITHUB_REPO, str(REPO_DIR)], check=True)

if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

import torch

vram_gb = (
    torch.cuda.get_device_properties(0).total_memory / (1024**3)
    if torch.cuda.is_available()
    else 0.0
)
USE_4BIT = vram_gb < 36.0
print(f"GPU VRAM: {vram_gb:.1f} GB -> use_4bit={USE_4BIT}")

hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    try:
        from google.colab import userdata

        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        hf_token = None
if hf_token:
    from huggingface_hub import login

    login(token=hf_token, add_to_git_credential=False)

if DATA_SOURCE_MODE == "google_drive":
    from google.colab import drive

    drive.mount("/content/drive")
    DATA_INPUT_DIR = DRIVE_DATA_DIR
elif DATA_SOURCE_MODE == "hf_dataset":
    from huggingface_hub import snapshot_download

    assert DATA_HF_REPO, "Set DATA_HF_REPO when DATA_SOURCE_MODE == 'hf_dataset'"
    DATA_INPUT_DIR = Path(snapshot_download(repo_id=DATA_HF_REPO, repo_type="dataset", local_dir=str(WORK_DIR / "hf_data")))
else:
    UPLOAD_DATA_DIR.mkdir(parents=True, exist_ok=True)
    DATA_INPUT_DIR = UPLOAD_DATA_DIR

print(f"DATA_SOURCE_MODE={DATA_SOURCE_MODE} -> DATA_INPUT_DIR={DATA_INPUT_DIR}")

## 2. Nạp/Kiểm duyệt Dataset & Đo Token Length thật (65,536 tokens)

In [ ]:
import json
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer
from distill.validator import convert_and_validate_openai_jsonl
from train.check_token_lengths import measure_dataset_token_lengths

WORK_DIR.mkdir(parents=True, exist_ok=True)
BASE_CKPT_DIR = Path(snapshot_download(repo_id=BASE_HF_REPO, local_dir=str(WORK_DIR / "base_ckpt")))
tok = AutoTokenizer.from_pretrained(str(BASE_CKPT_DIR), use_fast=True)

COMPILED_DIR = WORK_DIR / "compiled_data"
SFT_JSONL = COMPILED_DIR / "sft_gemma4.jsonl"
ORPO_JSONL = COMPILED_DIR / "orpo_gemma4.jsonl"
REJECTED_JSONL = COMPILED_DIR / "rejected.jsonl"

pre_sft = DATA_INPUT_DIR / "sft_gemma4.jsonl"
pre_orpo = DATA_INPUT_DIR / "orpo_gemma4.jsonl"
raw_sft = next((p for p in (DATA_INPUT_DIR / "sft_final_1350.jsonl", DATA_INPUT_DIR / "sft_merged.jsonl", REPO_DIR / "data" / "distill" / "sft_final_1350.jsonl") if p.exists()), None)
raw_orpo = next((p for p in (DATA_INPUT_DIR / "pref_final_270.jsonl", DATA_INPUT_DIR / "pref_merged.jsonl", REPO_DIR / "data" / "distill" / "pref_final_270.jsonl") if p.exists()), None)

if pre_sft.exists() and pre_orpo.exists():
    COMPILED_DIR.mkdir(parents=True, exist_ok=True)
    SFT_JSONL.write_text(pre_sft.read_text(encoding="utf-8"), encoding="utf-8")
    ORPO_JSONL.write_text(pre_orpo.read_text(encoding="utf-8"), encoding="utf-8")
    print(f"Loaded pre-compiled Gemma 4 JSONLs from {DATA_INPUT_DIR}")
else:
    raw_inputs = [str(p) for p in (raw_sft, raw_orpo) if p is not None]
    assert raw_inputs, f"Upload sft_gemma4.jsonl+orpo_gemma4.jsonl or sft_final_1350.jsonl+pref_final_270.jsonl into {DATA_INPUT_DIR}"
    ingest_report = convert_and_validate_openai_jsonl(
        raw_jsonl_paths=raw_inputs,
        out_sft_path=SFT_JSONL,
        out_orpo_path=ORPO_JSONL,
        out_rejected_path=REJECTED_JSONL,
        tokenizer=tok,
        max_tokens=SFT_MAX_SEQ_LENGTH,
    )
    print(json.dumps(ingest_report, indent=2, ensure_ascii=False))
    assert ingest_report["kept_sft"] > 0 and ingest_report["rejected_count"] == 0

tok_report = measure_dataset_token_lengths(
    SFT_JSONL,
    ORPO_JSONL,
    tokenizer=tok,
    sft_limit=SFT_MAX_SEQ_LENGTH,
    orpo_limit=ORPO_MAX_LENGTH,
    orpo_prompt_limit=ORPO_MAX_PROMPT_LENGTH,
)
print(json.dumps(tok_report, indent=2, ensure_ascii=False))
assert tok_report["unk_count"] == 0, f"Found {tok_report['unk_count']} <unk> tokens!"
assert tok_report["fits_100_pct"], "Adjust SFT_MAX_SEQ_LENGTH / ORPO_MAX_LENGTH to fit 100% of samples!"

## 3. Stage 2A — QLoRA/BF16 SFT (Vision/Audio Frozen, Low-LR Embeddings/PLE)

In [ ]:
from train.train_sft import SFTConfig, run_sft_training

SFT_MERGED_DIR = WORK_DIR / "stage2a_sft_merged"
if RUN_SFT:
    sft_cfg = SFTConfig(
        base_model_id=str(BASE_CKPT_DIR),
        train_jsonl=str(SFT_JSONL),
        output_dir=str(WORK_DIR / "stage2a_sft_lora"),
        merged_dir=str(SFT_MERGED_DIR),
        max_seq_length=SFT_MAX_SEQ_LENGTH,
        use_4bit=USE_4BIT,
        train_embeddings=TRAIN_EMBEDDINGS,
    )
    sft_summary = run_sft_training(sft_cfg)
    print(json.dumps(sft_summary, indent=2, ensure_ascii=False))

## 4. Stage 2B — 1-Epoch ORPO Alignment (Embeddings/PLE + Vision/Audio Frozen)

In [ ]:
from train.train_orpo import ORPOConfig, run_orpo_training

FINAL_MERGED_DIR = WORK_DIR / "stage2b_orpo_merged"
if RUN_ORPO:
    orpo_cfg = ORPOConfig(
        sft_merged_dir=str(SFT_MERGED_DIR),
        orpo_jsonl=str(ORPO_JSONL),
        output_dir=str(WORK_DIR / "stage2b_orpo_lora"),
        merged_dir=str(FINAL_MERGED_DIR),
        max_length=ORPO_MAX_LENGTH,
        max_prompt_length=ORPO_MAX_PROMPT_LENGTH,
        use_4bit=USE_4BIT,
        beta=0.1,
        learning_rate=8e-6,
    )
    orpo_summary = run_orpo_training(orpo_cfg)
    print(json.dumps(orpo_summary, indent=2, ensure_ascii=False))
else:
    FINAL_MERGED_DIR = SFT_MERGED_DIR

## 5. Export GGUF 4GB VRAM (`IQ4_XS` + `Q4_K_M` + `--mmproj bf16/q8_0`)

In [ ]:
import os
from train.export_gguf import DEFAULT_PLE_CPU_OFFLOAD, export_modular_gguf, write_imatrix_calib_corpus

if (LLAMA_CPP_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(LLAMA_CPP_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ggml-org/llama.cpp", str(LLAMA_CPP_DIR)], check=True)

subprocess.run(["cmake", "-B", str(LLAMA_CPP_DIR / "build"), "-S", str(LLAMA_CPP_DIR), "-DGGML_CUDA=OFF"], check=True)
subprocess.run(
    [
        "cmake",
        "--build",
        str(LLAMA_CPP_DIR / "build"),
        "--config",
        "Release",
        "--target",
        "llama-quantize",
        "llama-imatrix",
        "llama-cli",
        f"-j{os.cpu_count() or 4}",
    ],
    check=True,
)

GGUF_DIR = WORK_DIR / "gguf"
calib_txt = write_imatrix_calib_corpus(SFT_JSONL, WORK_DIR / "imatrix_calib.txt", max_samples=IMATRIX_SAMPLES) if USE_IMATRIX else None
export_manifest = export_modular_gguf(
    model_dir=FINAL_MERGED_DIR,
    out_dir=GGUF_DIR,
    llama_cpp_dir=LLAMA_CPP_DIR,
    model_prefix="gemma4-e4b-nc",
    quants=EXPORT_QUANTS,
    mmproj_outtypes=MMPROJ_OUTTYPES,
    imatrix_calib_file=calib_txt,
    imatrix_chunks=IMATRIX_SAMPLES,
)
print(json.dumps(export_manifest, indent=2, ensure_ascii=False))

cli_bin = LLAMA_CPP_DIR / "build" / "bin" / "llama-cli"
cli_help = subprocess.run([str(cli_bin), "--help"], capture_output=True, text=True).stdout
cli_st = ["--single-turn"] if "--single-turn" in cli_help else (["-no-cnv"] if "-no-cnv" in cli_help else [])
iq4_path = export_manifest["artifacts"][EXPORT_QUANTS[0].upper()]["path"]
smoke = subprocess.run(
    [
        str(cli_bin),
        "-m", iq4_path,
        "-ot", DEFAULT_PLE_CPU_OFFLOAD,
        "--jinja",
        "-p", "Viết 1 hàm Rust tính tổng i32:",
        "-n", "96",
        "--no-display-prompt",
        *cli_st,
    ],
    stdin=subprocess.DEVNULL,
    capture_output=True,
    text=True,
    check=True,
)
print("llama-cli smoke (IQ4_XS + PLE CPU offload):\n", smoke.stdout[:500])

## 6. Đẩy Thành Phẩm lên `haku-nekooo/gemma4-e4b-nuclear-agent`

In [ ]:
if PUSH_TO_HF:
    from huggingface_hub import HfApi

    api = HfApi()
    api.create_repo(repo_id=TARGET_HF_REPO, private=True, exist_ok=True)
    api.upload_folder(
        repo_id=TARGET_HF_REPO,
        folder_path=str(FINAL_MERGED_DIR),
        commit_message="Step 2: Upload Stage 2A/2B merged BF16 Gemma 4 E4B (Wave 1-4 PER Loop)",
    )
    api.upload_folder(
        repo_id=TARGET_HF_REPO,
        folder_path=str(GGUF_DIR),
        path_in_repo="gguf",
        commit_message="Step 2: Upload imatrix IQ4_XS/Q4_K_M + bf16/q8_0 mmproj GGUFs",
    )
    print(f"Uploaded artifacts -> https://huggingface.co/{TARGET_HF_REPO}")
else:
    print("PUSH_TO_HF=False -> skipped upload.")